In [1]:
!pip install -q langchain langchain-community langgraph chromadb sentence-transformers pypdf huggingface_hub

In [2]:
!pip install -q langchain-text-splitters

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import HuggingFaceHub

from langgraph.graph import StateGraph
from typing import TypedDict

import os

In [4]:
from google.colab import files
uploaded = files.upload()

Saving college_support.pdf to college_support (1).pdf


In [5]:
loader = PyPDFLoader("college_support.pdf")
documents = loader.load()

print("Total pages loaded:", len(documents))

Total pages loaded: 1


In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 3


In [7]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_11120/1474760240.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

retriever = vectorstore.as_retriever()

In [9]:
import getpass
os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass.getpass("Enter your HuggingFace API Key: ")

Enter your HuggingFace API Key: ··········


In [10]:
!pip install -q transformers accelerate

In [11]:
!pip install -q --upgrade transformers

In [14]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [15]:
def generate_answer(prompt):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

    outputs = model.generate(
        **inputs,
        max_length=256
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [17]:
def rag_pipeline(query):
    docs = retriever.invoke(query)
    context = "\n".join([doc.page_content for doc in docs])

    prompt = f"""
Answer the question using ONLY the context below.
Give a SHORT and DIRECT answer.

Context:
{context}

Question: {query}

Answer:
"""

    response = generate_answer(prompt)

    return response.strip()

In [18]:
query = "What is minimum attendance?"
print(rag_pipeline(query))

75%


In [19]:
print(rag_pipeline("What are exam rules?"))
print(rag_pipeline("What is grading system?"))
print(rag_pipeline("Can I get refund?"))

Students must carry their ID cards to the examination hall.
Grades are awarded as follows: O: 10 points A+: 9 points A+: 8 points B+: 7 points A+: 9 points B+: 7 points C: 5 points P: 4 points F: Fail
B


In [20]:
from typing import TypedDict

class GraphState(TypedDict):
    query: str
    context: str
    response: str
    confidence: float
    route: str

In [28]:
def process_node(state):
    query = state["query"]

    docs = retriever.invoke(query)
    context = "\n".join([doc.page_content for doc in docs])

    prompt = f"""
Answer the question using ONLY the context below.
Give a SHORT and DIRECT answer.

Context:
{context}

Question: {query}

Answer:
"""

    response = generate_answer(prompt)

    # simple confidence logic
    confidence = 0.9 if len(context) > 100 else 0.3

    return {
        "query": query,
        "context": context,
        "response": response,
        "confidence": confidence
    }

In [29]:
def route_node(state):
    if state["confidence"] < 0.5:
        return "hitl"
    else:
        return "output"

In [30]:
def hitl_node(state):
    print("⚠️ Escalating to Human...")

    # simulate human answer
    human_response = "Please contact the administration for more details."

    return {
        "query": state["query"],
        "context": state["context"],
        "response": human_response,
        "confidence": 1.0,
        "route": "hitl"
    }

In [31]:
def output_node(state):
    return state

In [32]:
from langgraph.graph import StateGraph

builder = StateGraph(GraphState)

# nodes
builder.add_node("process", process_node)
builder.add_node("hitl", hitl_node)
builder.add_node("output", output_node)

# edges
builder.set_entry_point("process")

builder.add_conditional_edges(
    "process",
    route_node,
    {
        "hitl": "hitl",
        "output": "output"
    }
)

builder.add_edge("hitl", "output")

graph = builder.compile()

In [33]:
result = graph.invoke({
    "query": "What is minimum attendance?"
})

print("Final Answer:", result["response"])
print("Route:", result.get("route", "auto"))

Final Answer: 75%
Route: auto


In [34]:
test_queries = [
    # ✅ Normal queries (should go AUTO)
    "What is minimum attendance?",
    "What are exam rules?",
    "Explain grading system",
    "What is the fee payment rule?",

    # ⚠️ Medium queries
    "What happens if attendance is low?",
    "Can I enter exam hall late?",

    # 🚨 Should trigger HITL (unknown / complex)
    "Can I get fee refund after 2 months?",
    "What is placement policy?",
    "Tell me hostel food menu",
    "Is there any scholarship available?"
]

for q in test_queries:
    print("\n" + "="*50)
    print("Query:", q)

    result = graph.invoke({"query": q})

    print("Answer:", result["response"])
    print("Route:", result.get("route", "auto"))


Query: What is minimum attendance?
Answer: 75%
Route: auto

Query: What are exam rules?
Answer: Students must carry their ID cards to the examination hall.
Route: auto

Query: Explain grading system
Answer: Grades are awarded as follows: O: 10 points A+: 9 points B+: 7 points A+: 9 points B+: 7 points A+: 9 points B+: 7 points C: 5 points P: 4 points F: Fail 4. Fee Payment Students must pay their semester fees before the deadline. Late payment may result in penalties. 5. Library Rules Students can borrow up to 3 books at a time for a period of 14 days. Late returns will incur a fine. 6. Hostel Rules Students must adhere to hostel timings. Entry after 10 PM is restricted unless prior permission is obtained. 7. Refund Policy Students can raise a support ticket.
Route: auto

Query: What is the fee payment rule?
Answer: B+
Route: auto

Query: What happens if attendance is low?
Answer: Students with attendance below 75% are not eligible to appear for exams
Route: auto

Query: Can I enter e